# Day 8: Snowflake Storage Integration & Dynamic Tables

This notebook uses DuckDB to simulate Snowflake's external storage querying and incremental dynamic table behavior.

In [ ]:
!pip install duckdb pandas
import duckdb
import pandas as pd

con = duckdb.connect(database=':memory:')
print("DuckDB connected.")

In [ ]:
# Simulate Iceberg / External Table Creation
con.execute("""
CREATE TABLE condition_occurrence_ext (
    condition_occurrence_id VARCHAR,
    person_id VARCHAR,
    condition_start_date DATE
);
""")

# Insert Base Data (Simulating S3 Parquet/Iceberg files)
con.execute("""
INSERT INTO condition_occurrence_ext VALUES 
('C001', 'P123', '2023-01-01'),
('C002', 'P124', '2023-01-05'),
('C003', 'P123', '2023-01-15') -- Readmission within 30 days
""")

print("--- BASE TABLE (Silver Zone) ---")
print(con.execute("SELECT * FROM condition_occurrence_ext").fetchdf())

# Simulate Dynamic Table (Materialized View in DuckDB)
con.execute("""
CREATE VIEW gold_readmissions AS
SELECT 
    person_id,
    COUNT(condition_occurrence_id) as admission_count
FROM condition_occurrence_ext
GROUP BY person_id
HAVING COUNT(condition_occurrence_id) > 1;
""")

print("\n--- DYNAMIC TABLE RESULT (Gold Zone) ---")
print(con.execute("SELECT * FROM gold_readmissions").fetchdf())

# Simulate new incremental data arriving
con.execute("""
INSERT INTO condition_occurrence_ext VALUES 
('C004', 'P124', '2023-01-20') -- P124 readmission
""")

print("\n--- DYNAMIC TABLE RESULT AFTER INCREMENTAL LOAD ---")
print(con.execute("SELECT * FROM gold_readmissions").fetchdf())
